# Speed Benchmark — Static Sparse Attention variants

Runs forward (and optionally backward) passes on real slides from the dataset
and reports throughput (slides/s) and latency (ms/slide) per configuration.

**How to add a variant:**  
Add an entry to `VARIANTS` — a dict with a readable `name` and a `factory`
callable that returns an `nn.Module` ready to call as `model(features, coords)`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import time
from pathlib import Path

import torch
import pandas as pd
from torch.utils.data import DataLoader

from sparse_wsi_vit.datasets.h5_slidedataset.h5_dataset import H5FeatureBagDataset
from sparse_wsi_vit.experiments.datamodules.h5_datamodule import mil_collate_fn
from sparse_wsi_vit.models.static_sparse_attention import StaticSparseViTSlideEncoder

## Config

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
CSV_PATH      = Path("../../splits/camelyon/0/val.csv")   # any split CSV
FEATURES_DIR  = "../../camelyon-emb/"
FEATURES_NAME = "cls_224x224"
COORDS_NAME   = "coords_224x224"
LABEL_COL     = "label"

# ── Benchmark settings ───────────────────────────────────────────────────────
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE         = torch.float32          # change to torch.bfloat16 to match bf16-mixed training
NUM_SLIDES    = 10                     # how many slides to time per variant (None = all)
WARMUP_SLIDES = 2                      # slides run before timing starts (compilation / caching)
WITH_BACKWARD = False                  # set True to include backward pass in timing
NUM_WORKERS   = 4

print(f"Device : {DEVICE}")
print(f"dtype  : {DTYPE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(DEVICE)}")

## Model variants

Each entry is a `dict` with:
- `name`    — label shown in the results table
- `factory` — zero-argument callable that returns an **un-compiled** model
- `compile` — whether to `torch.compile` the model before timing

Shared architecture kwargs live in `BASE_KWARGS` so they're not repeated.

In [ ]:
BASE_KWARGS = dict(
    in_features  = 1280,
    out_features = 1,
    embed_dim    = 384,
    num_heads    = 6,
    num_layers   = 6,
    num_cls      = 4,
    window_size  = 5,
    dilation     = 1,
    rope_theta   = 10_000.0,
    rope_coord_high = 100_000.0,
)

VARIANTS = [
    dict(
        name    = "chunk=512",
        factory = lambda: StaticSparseViTSlideEncoder(**BASE_KWARGS, chunk_size=512),
        compile = False,
    ),
    dict(
        name    = "chunk=4096",
        factory = lambda: StaticSparseViTSlideEncoder(**BASE_KWARGS, chunk_size=4096),
        compile = False,
    ),
    dict(
        name    = "chunk=32768",
        factory = lambda: StaticSparseViTSlideEncoder(**BASE_KWARGS, chunk_size=32_768),
        compile = False,
    ),
    # Uncomment to test compiled variants:
    # dict(
    #     name    = "chunk=32768 + compile",
    #     factory = lambda: StaticSparseViTSlideEncoder(**BASE_KWARGS, chunk_size=32_768),
    #     compile = True,
    # ),
]

## Dataset

In [ ]:
dataset = H5FeatureBagDataset(
    csv_path      = str(CSV_PATH),
    features_dir  = FEATURES_DIR,
    label_col_name = LABEL_COL,
    features_name = FEATURES_NAME,
    coords_name   = COORDS_NAME,
)

loader = DataLoader(
    dataset,
    batch_size  = 1,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    collate_fn  = mil_collate_fn,
    pin_memory  = (DEVICE.type == "cuda"),
)

n_slides = len(dataset)
print(f"Dataset: {n_slides} slides")
print(f"Timing on {NUM_SLIDES or n_slides} slides  (warmup: {WARMUP_SLIDES})")

## Benchmark loop

In [ ]:
def benchmark_variant(variant: dict, loader: DataLoader) -> dict:
    """Run one variant and return timing stats."""
    model = variant["factory"]()
    model = model.to(device=DEVICE, dtype=DTYPE)
    if variant.get("compile"):
        print(f"  compiling {variant['name']} …", end=" ", flush=True)
        model = torch.compile(model, mode="max-autotune-no-cudagraphs")

    model.eval() if not WITH_BACKWARD else model.train()

    slide_times   = []   # ms per slide
    patch_counts  = []   # patches per slide
    n_total       = NUM_SLIDES + WARMUP_SLIDES if NUM_SLIDES else None

    ctx = torch.no_grad() if not WITH_BACKWARD else torch.enable_grad()

    with ctx:
        for i, batch in enumerate(loader):
            if n_total and i >= n_total:
                break

            features = batch["input"].to(device=DEVICE, dtype=DTYPE)  # (1, N, D)
            coords   = batch["coords"].to(device=DEVICE)               # (1, N, 2)
            n_patches = features.shape[1]

            # ── timed section ────────────────────────────────────────────────
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            out = model(features, coords)
            loss = out["logits"].sum()   # scalar needed for backward
            if WITH_BACKWARD:
                loss.backward()

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            elapsed_ms = (time.perf_counter() - t0) * 1000
            # ────────────────────────────────────────────────────────────────

            if i < WARMUP_SLIDES:
                print(f"  [warmup {i+1}/{WARMUP_SLIDES}] {n_patches:,} patches — {elapsed_ms:.0f} ms")
                continue

            slide_times.append(elapsed_ms)
            patch_counts.append(n_patches)

    import statistics
    return {
        "name"         : variant["name"],
        "n_slides"     : len(slide_times),
        "mean_ms"      : statistics.mean(slide_times),
        "median_ms"    : statistics.median(slide_times),
        "min_ms"       : min(slide_times),
        "max_ms"       : max(slide_times),
        "mean_patches" : int(statistics.mean(patch_counts)),
        "slides_per_s" : 1000 / statistics.mean(slide_times),
    }


results = []
for v in VARIANTS:
    print(f"\n── {v['name']} {'(+compile)' if v.get('compile') else ''} ──")
    r = benchmark_variant(v, loader)
    results.append(r)
    print(f"  mean {r['mean_ms']:.0f} ms  |  median {r['median_ms']:.0f} ms  |  "
          f"{r['slides_per_s']:.2f} slides/s  |  {r['mean_patches']:,} patches/slide")

## Results table

In [ ]:
df = pd.DataFrame(results).set_index("name")
df["speedup_vs_first"] = df["mean_ms"].iloc[0] / df["mean_ms"]

display(df[["mean_ms", "median_ms", "min_ms", "max_ms", "slides_per_s", "mean_patches", "speedup_vs_first"]]
        .rename(columns={
            "mean_ms"          : "mean (ms)",
            "median_ms"        : "median (ms)",
            "min_ms"           : "min (ms)",
            "max_ms"           : "max (ms)",
            "slides_per_s"     : "slides/s",
            "mean_patches"     : "patches/slide",
            "speedup_vs_first" : "speedup vs first",
        })
        .style.format({
            "mean (ms)"       : "{:.0f}",
            "median (ms)"     : "{:.0f}",
            "min (ms)"        : "{:.0f}",
            "max (ms)"        : "{:.0f}",
            "slides/s"        : "{:.3f}",
            "patches/slide"   : "{:,d}",
            "speedup vs first": "{:.2f}×",
        })
        .background_gradient(subset=["mean (ms)"], cmap="RdYlGn_r")
       )

## Per-slide breakdown

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Latency distribution per variant
ax = axes[0]
for r in results:
    ax.plot(r["mean_ms"], marker="o", label=r["name"])
ax.set_title("Mean latency (ms / slide)")
ax.set_ylabel("ms")
ax.legend()
ax.grid(True, alpha=0.3)

# Throughput bar chart
ax = axes[1]
names  = [r["name"] for r in results]
speeds = [r["slides_per_s"] for r in results]
bars   = ax.bar(names, speeds, color="steelblue")
ax.bar_label(bars, fmt="{:.3f}", padding=3)
ax.set_title("Throughput (slides / s)")
ax.set_ylabel("slides/s")
ax.set_ylim(0, max(speeds) * 1.25)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()